# Install Additional Libraries

In [1]:
! pip install mldesigner # required for creating pipelines

# Imports

In [2]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

# Authenticate

In [3]:
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    # subscription_id="[REDACTED_SUBSCRIPTION_ID]", # your subscription id
    # workspace_name="[REDACTED_WORKSPACE]", # your workspace name
    # resource_group_name="[REDACTED_RESOURCE_GROUP]" # your resource group name
)

# Creating a Custom Reusable Environment

Before we begin creating a pipeline, we need to create an environment with all the required libraries.  
We can also use this same environment for deployment purposes later on

#### Define a `yaml` file with the environment configurations

- This is very similar to a requirements.txt file in python. We will add a few libraries that are required for deployment while creating this environment so that we can reuse it without having to create another environment or update this again
- We can use the `%%writefile` magic command in jupyter notebook to write the contents of a cell to a file called `environment-config.yaml`.  
- Use this link to learn more about magic commands in jupyter notebook: https://medium.com/@marc.bolle/learn-these-15-magic-commands-in-jupyter-notebook-to-save-time-a864ca9b15c7


In [4]:
%%writefile scripts/environment-config.yaml
name: clf-model-environment-pipeline
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - numpy
      - pandas
      - scikit-learn
      - mlflow==2.17.0
      - azureml-mlflow==1.58.0
      - azure-identity
      - azureml
      - azureml-core
      - azure-ai-ml
      - azureml-inference-server-http
      - fsspec
      - azureml-fsspec
      - mldesigner

Writing scripts/environment-config.yaml


#### Create the environment

In [5]:
from azure.ai.ml.entities import Environment

In [6]:
my_environment = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04", # use a base image from microsoft container registry
    conda_file="scripts/environment-config.yaml", # name of the environment yaml file we just created
    name="telecom-churn-environment",
    description="Environment for building telecom churn classification models",
)

env_create = ml_client.environments.create_or_update(my_environment)

In [7]:
env_target = ml_client.environments.get('telecom-churn-environment', label='latest')
print(env_target)

conda_file:
  channels:
  - conda-forge
  dependencies:
  - python=3.10
  - pip
  - pip:
    - numpy
    - pandas
    - scikit-learn
    - mlflow==2.17.0
    - azureml-mlflow==1.58.0
    - azure-identity
    - azureml
    - azureml-core
    - azure-ai-ml
    - azureml-inference-server-http
    - fsspec
    - azureml-fsspec
    - mldesigner
  name: clf-model-environment-pipeline
creation_context:
  created_at: '2026-10-17T20:51:50.566266+00:00'
  created_by: [REDACTED_NAME]
  created_by_type: CreatedByType.USER
  last_modified_at: '2026-10-17T20:51:50.566266+00:00'
  last_modified_by: [REDACTED_NAME]
  last_modified_by_type: CreatedByType.USER
description: Environment for building telecom churn classification models
id: azureml:/subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourceGroups/[REDACTED_RESOURCE_GROUP]/providers/Microsoft.MachineLearningServices/workspaces/[REDACTED_WORKSPACE]/environments/telecom-churn-environment/versions/1
image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20

# Creating ML Pipeline

##### Creating the `Data Preparation Step/Component` for the Pipeline

We will do the following two things to create a component
- We will convert our previous data prep code into a function with inputs and outputs defined
- We will use the `@command_component` function decorator available through the `mldesigner` library to convert our function into a pipeline component 


Note on Decorators in Python: https://www.datacamp.com/tutorial/decorators-python

In [8]:
%%writefile scripts/data_prep_component.py

import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split

# import the library to help build the pipeline components
from mldesigner import command_component, Input, Output

@command_component(
    name="data_prep_component", # name of the component or step
    display_name="Data Preparation", # more readable name of the component or step
    description="Data Preparation - Split telecom churn data into training and test sets",
    environment="azureml:telecom-churn-environment@latest"
)
def data_prep_component(
    input_data: Input(type="uri_file"),
    x_train: Output(type="uri_file"),
    x_test: Output(type="uri_file"),
    y_train: Output(type="uri_file"),
    y_test: Output(type="uri_file")
):
    with mlflow.start_run(run_name="data preparation run") as run:
        df = pd.read_csv(input_data)

        # remove duplicate records
        df = df.drop_duplicates()

        # create the additional derived metric from Question 2
        df["AvgDayMinsPerCall"] = df["DayMins"] / df["DayCalls"].replace(0, 1)

        # perform train-test split
        test_size = 0.3
        target_column_name = "Churn"

        X_train, X_test, Y_train, Y_test = train_test_split(
            df[df.columns.difference([target_column_name]).values],
            df[target_column_name],
            stratify=df[target_column_name],
            test_size=test_size,
            random_state=123456
        )

        # log metrics and parameters with mlflow
        mlflow.log_metric("Test Set Size", test_size)
        mlflow.log_param("Target Label Column Name", target_column_name)
        mlflow.log_param("Training Data Shape", str(X_train.shape))
        mlflow.log_param("Test Data Shape", str(X_test.shape))
        mlflow.log_param("Preprocessing Transformations", "Removed duplicates; handled zero daytime calls")
        mlflow.log_param("New Feature Added", "AvgDayMinsPerCall")

        # save the output files for consumption by another node
        X_train.to_csv(x_train, index=False)
        X_test.to_csv(x_test, index=False)
        Y_train.to_csv(y_train, index=False)
        Y_test.to_csv(y_test, index=False)

Writing scripts/data_prep_component.py


##### Creating the `Model Training and Evaluation Step/Component` for the Pipeline

In [9]:
%%writefile scripts/train_model_component.py

import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score, precision_score

# import the library to help build the pipeline components
from mldesigner import command_component, Input, Output

@command_component(
    name="train_model_component", # name of the component or step
    display_name="Model Training", # more readable name of the component or step
    description="Model Training - Train an optimized Random Forest classifier",
    environment="azureml:telecom-churn-environment@latest"
)
def train_model_component(
    x_train: Input(type="uri_file"),
    x_test: Input(type="uri_file"),
    y_train: Input(type="uri_file"),
    y_test: Input(type="uri_file"),
    model_output: Output(type="uri_folder")
):
    X_train = pd.read_csv(x_train)
    X_test = pd.read_csv(x_test)
    Y_train = pd.read_csv(y_train).iloc[:, 0]
    Y_test = pd.read_csv(y_test).iloc[:, 0]

    # this will help us use mlflow for automatic logging
    mlflow.sklearn.autolog()

    # train the optimized Random Forest model from Question 4
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=123456
    )

    # the model artifact will be automatically logged
    model.fit(X_train, Y_train)

    # calculate accuracy
    acc = model.score(X_test, Y_test)
    print("Test Accuracy:", acc)

    # calculate and log additional evaluation metrics
    predictions = model.predict(X_test)

    f1 = f1_score(Y_test, predictions)
    precision = precision_score(Y_test, predictions)

    mlflow.log_metric("Test F1 Score", f1)
    mlflow.log_metric("Test Precision", precision)

    print("Test F1 Score:", f1)
    print("Test Precision:", precision)

    # log plots for model performance using mlflow
    cm = confusion_matrix(Y_test, predictions, labels=model.classes_)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=model.classes_
    )
    mlflow.log_figure(disp.plot().figure_, "test_set_confusion_matrix.png")

    # save model to output node
    mlflow.sklearn.save_model(model, model_output)

Writing scripts/train_model_component.py


##### Combine the Components and Create the `End-to-End Pipeline`

In [10]:
# load the components
from scripts.data_prep_component import data_prep_component
from scripts.train_model_component import train_model_component

# pipeline builder
from azure.ai.ml.dsl import pipeline

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.16.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# define the pipeline
@pipeline(
    name='telecom_churn_mlops_pipeline',
    description="Data preparation and optimized Random Forest model training pipeline",
    default_compute="amlcomputeops0001" # compute resource to be used to run the pipeline
)
def telecom_churn_pipeline(pipeline_input_data):
    # step 1 - data prep step
    data_prep_node = data_prep_component(
        input_data=pipeline_input_data
    )

    # step 2 - model training and evaluation step
    train_model_node = train_model_component(
        x_train=data_prep_node.outputs.x_train,
        x_test=data_prep_node.outputs.x_test,
        y_train=data_prep_node.outputs.y_train,
        y_test=data_prep_node.outputs.y_test
    )

In [15]:
# input dataset for the pipeline
my_input_dataset = ml_client.data.get('telecom_churn_dataset_mlops', version='v2')

# create the pipeline
pipeline_job = telecom_churn_pipeline(pipeline_input_data=my_input_dataset)
print(pipeline_job)

pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="mini-project-2-telecom-churn-pipeline"
)
print(pipeline_job)

# wait until the pipeline job completes
ml_client.jobs.stream(pipeline_job.name)

display_name: telecom_churn_mlops_pipeline
description: Data preparation and optimized Random Forest model training pipeline
type: pipeline
inputs:
  pipeline_input_data:
    type: uri_file
    path: azureml:/subscriptions/[REDACTED_SUBSCRIPTION_ID]/resourceGroups/[REDACTED_RESOURCE_GROUP]/providers/Microsoft.MachineLearningServices/workspaces/[REDACTED_WORKSPACE]/data/telecom_churn_dataset_mlops/versions/v2
jobs:
  data_prep_node:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_input_data}}
    component:
      name: data_prep_component
      display_name: Data Preparation
      description: Data Preparation - Split telecom churn data into training and test
        sets
      tags:
        codegenBy: mldesigner
      type: command
      inputs:
        input_data:
          type: uri_file
      outputs:
        x_train:
          type: uri_file
        x_test:
          type: uri_file
        y_train:
          type: uri_file
        y_test:
   

Uploading scripts (0.01 MBs): 100%|██████████| 5137/5137 [00:00<00:00, 72081.40it/s]


